In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.chdir("/content/drive/MyDrive/pokemon")

In [ ]:
!pip install pyboy keyboard numpy

: 

In [ ]:
import numpy as np


class SumTree:
    data_pointer = 0

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1)
        self.data = np.zeros(capacity, dtype=object)

    def add(self, p: float, data):
        tree_idx = self.data_pointer + self.capacity - 1
        self.data[self.data_pointer] = data
        self.update(tree_idx, p)

        self.data_pointer += 1
        if self.data_pointer >= self.capacity:
            self.data_pointer = 0

    def update(self, tree_idx: int, p: float):
        change = p - self.tree[tree_idx]
        self.tree[tree_idx] = p
        while tree_idx != 0:
            tree_idx = (tree_idx - 1) // 2
            self.tree[tree_idx] += change

    def get_leaf(self, v: float):
        parent_idx = 0
        while True:
            left_child_idx = 2 * parent_idx + 1
            right_child_idx = left_child_idx + 1

            if left_child_idx >= len(self.tree):
                leaf_idx = parent_idx
                break

            if v <= self.tree[left_child_idx]:
                parent_idx = left_child_idx
            else:
                v -= self.tree[left_child_idx]
                parent_idx = right_child_idx

        data_idx = leaf_idx - self.capacity + 1
        return leaf_idx, self.tree[leaf_idx], self.data[data_idx]

    @property
    def total_p(self):
        return self.tree[0]


: 

In [ ]:
import random
import numpy as np


class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, alpha=0.6):
        self.tree = SumTree(capacity)
        self.alpha = alpha
        self.epsilon = 0.01

    def add(self, data):
        max_p = np.max(self.tree.tree[-self.tree.capacity :])
        if max_p == 0:
            max_p = 1.0
        self.tree.add(max_p, data)

    def sample(self, n: int, beta=0.4):
        batch = []
        idxs = []
        segment = self.tree.total_p / n
        priorities = []

        self.beta = beta

        for i in range(n):
            a = segment * i
            b = segment * (i + 1)
            s = random.uniform(a, b)
            (idx, p, data) = self.tree.get_leaf(s)

            if data is None:
                (idx, p, data) = self.tree.get_leaf(a)

            priorities.append(p)
            batch.append(data)
            idxs.append(idx)

        sampling_probabilities = np.array(priorities) / self.tree.total_p
        is_weights = np.power(self.tree.capacity * sampling_probabilities, -self.beta)
        is_weights /= is_weights.max()

        return batch, idxs, is_weights

    def update_priorities(self, idxs: list[int], errors: list[float]):
        for idx, error in zip(idxs, errors):
            p = (error + self.epsilon) ** self.alpha
            self.tree.update(idx, p)

    def __len__(self) -> int:
        return self.tree.data_pointer


: 

In [ ]:
from dataclasses import dataclass, field

from pyboy import PyBoy, PyBoyMemoryView
import torch


@dataclass
class Data:
    pyboy: PyBoy

    visited_dialogs_count: dict[int, int] = field(default_factory=dict)
    visited_dialogs_count_max: int = 8
    visited_positions_count: dict[str, int] = field(default_factory=dict)
    visited_maps_count: dict[int, int] = field(default_factory=dict)
    visited_maps_count_max: int = 4
    useless_count: int = 0
    __player_pokemon_size = 0x2C
    __pokemon_count = 6

    __stored_pokemon_size = 0x21
    __stored_pokemon_count = 20

    __visited_pokedex_own: list[int] | None = None

    @property
    def visited_pokedex_own(self):
        if self.__visited_pokedex_own is None:
            self.__visited_pokedex_own = self.pokedex_own(self.pyboy.memory)

        return self.__visited_pokedex_own

    @visited_pokedex_own.setter
    def visited_pokedex_own(self, value: list[int]):
        self.__visited_pokedex_own = [
            x | y for x, y in zip(value, self.visited_pokedex_own)
        ]

    __visited_pokedex_seen: list[int] | None = None

    @property
    def visited_pokedex_seen(self):
        if self.__visited_pokedex_seen is None:
            self.__visited_pokedex_seen = self.pokedex_seen(self.pyboy.memory)

        return self.__visited_pokedex_seen

    @visited_pokedex_seen.setter
    def visited_pokedex_seen(self, value: list[int]):
        self.__visited_pokedex_seen = [
            x | y for x, y in zip(value, self.visited_pokedex_seen)
        ]

    def clean(self):
        self.__visited_pokedex_own = None
        self.__visited_pokedex_seen = None
        self.visited_dialogs_count = {}
        self.visited_maps_count = {}
        self.visited_positions_count = {}
        self.useless_count = 0

    def count(self, memory: bytes, reward: float):
        if self.is_dialog(self.pyboy.memory):
            self.visited_dialogs_count.setdefault(self.dialog_id(self.pyboy.memory), 0)
            self.visited_dialogs_count[self.dialog_id(self.pyboy.memory)] += 1

        if self.is_world(self.pyboy.memory):
            self.visited_positions_count.setdefault(self.get_position(), 0)
            self.visited_positions_count[self.get_position()] += 1

        self.visited_pokedex_own = self.pokedex_own(self.pyboy.memory)
        self.visited_pokedex_seen = self.pokedex_seen(self.pyboy.memory)

        self.visited_maps_count.setdefault(self.map_id(self.pyboy.memory), 0)
        self.visited_maps_count[self.map_id(self.pyboy.memory)] += 1

        if 0 < reward:
            self.useless_count = 0
        else:
            self.useless_count += 1

    def inputs(self):
        return {
            "continuous": torch.tensor(self.data(), dtype=torch.float32),
            "map_id": torch.tensor(self.map_id(self.pyboy.memory), dtype=torch.long),
            "dialog_id": torch.tensor(
                self.dialog_id(self.pyboy.memory), dtype=torch.long
            ),
            "index_of_current_pokemon_send_out": torch.tensor(
                self.index_of_current_pokemon_send_out(self.pyboy.memory),
                dtype=torch.long,
            ),
            "type_of_battle": torch.tensor(
                self.type_of_battle(self.pyboy.memory),
                dtype=torch.long,
            ),
            "move_menu_type": torch.tensor(
                self.move_menu_type(self.pyboy.memory),
                dtype=torch.long,
            ),
            "position_x": torch.tensor(
                self.position_x(self.pyboy.memory), dtype=torch.float32
            ),
            "position_y": torch.tensor(
                self.position_y(self.pyboy.memory), dtype=torch.float32
            ),
            "bike_speed": torch.tensor(
                self.bike_speed(self.pyboy.memory), dtype=torch.float32
            ),
            "menu_position_x": torch.tensor(
                self.menu_position_x(self.pyboy.memory), dtype=torch.long
            ),
            "menu_position_y": torch.tensor(
                self.menu_position_y(self.pyboy.memory), dtype=torch.long
            ),
            "current_menu_selected_item": torch.tensor(
                self.current_menu_selected_item(self.pyboy.memory), dtype=torch.long
            ),
            "visited_dialogs_count": torch.tensor(
                (
                    min(
                        self.visited_dialogs_count.get(
                            self.dialog_id(self.pyboy.memory), 0
                        ),
                        self.visited_dialogs_count_max,
                    )
                    if self.is_dialog(self.pyboy.memory)
                    else 0
                ),
                dtype=torch.long,
            ),
            "visited_maps_count": torch.tensor(
                (
                    min(
                        self.visited_maps_count.get(self.map_id(self.pyboy.memory), 0),
                        self.visited_maps_count_max,
                    )
                    if self.is_world(self.pyboy.memory)
                    else 0
                ),
                dtype=torch.long,
            ),
            "move_id": torch.tensor(
                [
                    self.player_selected_move(self.pyboy.memory),
                    self.enemy_selected_move(self.pyboy.memory),
                    self.enemy_move1(self.pyboy.memory),
                    self.enemy_move2(self.pyboy.memory),
                    self.enemy_move3(self.pyboy.memory),
                    self.enemy_move4(self.pyboy.memory),
                ]
                + self.stored_pokemon_moves(self.pyboy.memory),
                dtype=torch.long,
            ),
            "move_type": torch.tensor(
                [
                    self.your_move_type(self.pyboy.memory),
                    self.enemy_move_type(self.pyboy.memory),
                    self.pokemon_move_first_slot(self.pyboy.memory),
                    self.pokemon_move_second_slot(self.pyboy.memory),
                    self.pokemon_move_third_slot(self.pyboy.memory),
                    self.pokemon_move_fourth_slot(self.pyboy.memory),
                ],
                dtype=torch.long,
            ),
            "pokemon_id": torch.tensor(
                self.player_pokemons_ids(self.pyboy.memory)
                + self.stored_pokemon_ids(self.pyboy.memory),
                dtype=torch.long,
            ),
            "pokemon_type": torch.tensor(
                [
                    self.enemy_type1(self.pyboy.memory),
                    self.enemy_type2(self.pyboy.memory),
                    self.pokemon_type1(self.pyboy.memory),
                    self.pokemon_type2(self.pyboy.memory),
                ]
                + self.player_pokemon_types(self.pyboy.memory)
                + self.stored_pokemon_types(self.pyboy.memory),
                dtype=torch.long,
            ),
            "sprite_id": torch.tensor(
                self.sprite_data_ids(self.pyboy.memory),
                dtype=torch.long,
            ),
            "item_id": torch.tensor(
                self.poke_mart_items(self.pyboy.memory)
                + self.items_ids(self.pyboy.memory)
                + self.stored_items_ids(self.pyboy.memory),
                dtype=torch.long,
            ),
            "sprite_data_movement_statuses": torch.tensor(
                self.sprite_data_movement_statuses(self.pyboy.memory),
                dtype=torch.long,
            ),
            "sprite_data_facing_directions": torch.tensor(
                self.sprite_data_facing_directions(self.pyboy.memory),
                dtype=torch.long,
            ),
            "sprite_data_y_positions": torch.tensor(
                self.sprite_data_y_positions(self.pyboy.memory),
                dtype=torch.long,
            ),
            "sprite_data_x_positions": torch.tensor(
                self.sprite_data_x_positions(self.pyboy.memory),
                dtype=torch.long,
            ),
        }

    def reward(self, memory: bytes):
        reward = 0.0

        reward += self.reward_core(memory)

        if self.is_battle(self.pyboy.memory):
            reward += self.reward_battle(memory)

        if self.is_dialog(self.pyboy.memory):
            reward += self.reward_dialog()

        if self.is_world(self.pyboy.memory):
            reward += self.reward_position()

        return max(-1.0, min(1.0, reward))

    def reward_core(self, memory: bytes):
        reward = 0.0

        reward += self.reward_milestones(memory)
        reward += self.reward_pokedex(memory)
        reward += self.reward_player_pokemons_current_hps(memory)
        reward += self.reward_player_pokemons_statuses(memory)
        reward += self.reward_player_pokemons_experiences(memory)
        reward += self.reward_player_pokemons_max_hps(memory)
        reward += self.reward_player_pokemons_attacks(memory)
        reward += self.reward_player_pokemons_defenses(memory)
        reward += self.reward_player_pokemons_speeds(memory)
        reward += self.reward_player_pokemons_pps(memory)
        reward += self.reward_map(memory)

        return reward

    def reward_map(self, memory: bytes):
        return 0.01 if self.visited_maps_count.get(self.map_id(memory), 0) < 4 else 0.0

    def reward_player_pokemons_current_hps(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, hp_x, hp_y, max_hp in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_current_hps(memory),
            self.player_pokemons_current_hps(self.pyboy.memory),
            self.player_pokemons_max_hps(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (hp_y - hp_x) / max_hp

        return reward

    def reward_player_pokemons_statuses(self, memory: bytes):
        ids_x = self.player_pokemons_ids(memory)
        ids_y = self.player_pokemons_ids(self.pyboy.memory)
        statuses_x = self.player_pokemons_statuses(memory)
        statuses_y = self.player_pokemons_statuses(self.pyboy.memory)
        statuses_x = [statuses_x[i : i + 7] for i in range(0, len(statuses_x), 7)]
        statuses_y = [statuses_y[i : i + 7] for i in range(0, len(statuses_y), 7)]

        reward = 0.0
        for id_x, id_y, status_x, status_y in zip(ids_x, ids_y, statuses_x, statuses_y):
            if id_x == id_y and id_x != 0:
                for status_x_bit, status_y_bit in zip(status_x, status_y):
                    if status_x_bit == 1 and status_y_bit == 0:
                        reward += 0.3
                    elif status_x_bit == 0 and status_y_bit == 1:
                        reward -= 0.3

        return reward

    def reward_player_pokemons_experiences(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, experience_x, experience_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_experiences(memory),
            self.player_pokemons_experiences(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (experience_y - experience_x) / 0xFFFFFF

        return reward

    def reward_player_pokemons_max_hps(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, max_hp_x, max_hp_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_max_hps(memory),
            self.player_pokemons_max_hps(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (max_hp_y - max_hp_x) / 0xFFFF

        return reward

    def reward_player_pokemons_attacks(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, attack_x, attack_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_attacks(memory),
            self.player_pokemons_attacks(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (attack_y - attack_x) / 0xFFFF

        return reward

    def reward_player_pokemons_defenses(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, defense_x, defense_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_defenses(memory),
            self.player_pokemons_defenses(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (defense_y - defense_x) / 0xFFFF

        return reward

    def reward_player_pokemons_speeds(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, speed_x, speed_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_speeds(memory),
            self.player_pokemons_speeds(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (speed_y - speed_x) / 0xFFFF

        return reward

    def reward_player_pokemons_specials(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, special_x, special_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_specials(memory),
            self.player_pokemons_specials(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (special_y - special_x) / 0xFFFF

        return reward

    def reward_player_pokemons_pps(self, memory: bytes):
        reward = 0.0

        for id_x, id_y, pp_x, pp_y in zip(
            self.player_pokemons_ids(memory),
            self.player_pokemons_ids(self.pyboy.memory),
            self.player_pokemons_pps(memory),
            self.player_pokemons_pps(self.pyboy.memory),
        ):
            if id_x == id_y and id_x != 0:
                reward += (pp_y - pp_x) / 0xFF

        return reward

    def terminated(self, memory: bytes):
        return True if 0 < self.reward_badges(memory) else False

    def truncated(self):
        return True if 128 <= self.useless_count else False

    def number_of_turns_in_current_battle(self, memory: bytes):
        return memory[0xCCD5]

    def data(self):
        data = []

        data += self.core_data()
        data += self.battle_data()
        data += self.world_data()

        return data

    def world_data(self):
        return [
            (
                min(self.visited_positions_count.get(self.get_position(), 0), 1)
                if self.is_world(self.pyboy.memory)
                else 0
            )
        ]

    def game_mode_flags_data(self):
        return [
            int(self.is_battle(self.pyboy.memory)),
            int(self.is_dialog(self.pyboy.memory)),
            int(self.is_menu(self.pyboy.memory)),
            int(self.is_world(self.pyboy.memory)),
        ]

    def data_normalizer(self, values: list[int], max=0xFF):
        return [x / max for x in values]

    def core_data(self):
        data = []
        data += [
            0 if self.visited_positions_count.get(self.get_position(), 0) == 0 else 1
        ]
        data += self.game_mode_flags_data()
        data += self.player_data()
        data += self.pokedex_data()
        data += self.data_normalizer(self.items_quantities(self.pyboy.memory))
        data += self.data_normalizer(
            [self.player_money(self.pyboy.memory)], max=0xFFFFFF
        )
        data += self.badges(self.pyboy.memory)
        data += self.data_normalizer(self.stored_items_quantities(self.pyboy.memory))
        data += self.data_normalizer([self.game_coins(self.pyboy.memory)], max=0xFFFF)
        data += self.event_flags_data(self.pyboy.memory)
        data += self.stored_pokemon_data(self.pyboy.memory)

        return data

    def map_id(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD35E]

    def is_dialog(self, memory: PyBoyMemoryView | bytes):
        return (
            True
            if self.is_blocked(memory)
            and self.dialog_id(memory) != 0
            and not self.is_battle(memory)
            else False
        )

    def is_blocked(self, memory: PyBoyMemoryView | bytes):
        return True if memory[0xCFC4] else False

    def dialog_id(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCF13]

    def is_battle(self, memory: PyBoyMemoryView | bytes):
        return True if self.type_of_battle(memory) else False

    def type_of_battle(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD057]

    def get_position(self):
        return f"{self.position_x(self.pyboy.memory)}x{self.position_y(self.pyboy.memory)}x{self.map_id(self.pyboy.memory)}"

    def is_world(self, memory: PyBoyMemoryView | bytes):
        return (
            True
            if not self.is_blocked(memory)
            and not self.is_battle(memory)
            and not self.is_menu(memory)
            else False
        )

    def position_x(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD361]

    def position_y(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD362]

    def is_menu(self, memory: PyBoyMemoryView | bytes):
        return (
            True
            if self.is_blocked(memory)
            and self.dialog_id(memory) == 0
            and not self.is_battle(memory)
            else False
        )

    def sprite_data_ids(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xC100 + 0x10 * x] for x in range(16)]

    def sprite_data_movement_statuses(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xC101 + 0x10 * x] for x in range(16)]

    def sprite_data_facing_directions(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xC109 + 0x10 * x] for x in range(16)]

    def sprite_data_y_positions(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xC204 + 0x10 * x] for x in range(16)]

    def sprite_data_x_positions(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xC205 + 0x10 * x] for x in range(16)]

    def menu_position_x(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCC24]

    def menu_position_y(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCC25]

    def current_menu_selected_item(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCC26]

    def index_of_current_pokemon_send_out(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCC2F]

    def battle_data(self):
        data_bit = (
            self.enemy_status(self.pyboy.memory)
            + self.enemy_base_stats(self.pyboy.memory)
            + self.pokemon_status(self.pyboy.memory)
            + self.battle_status_player(self.pyboy.memory)
            + [
                self.is_gym_leader_battle_music_playing(self.pyboy.memory),
                self.critical_hit_flag(self.pyboy.memory),
                self.one_hit_ko_flag(self.pyboy.memory),
                self.hooked_pokemon_flag(self.pyboy.memory),
            ]
        )
        data_byte = self.data_normalizer(
            [
                self.players_substitute_hp(self.pyboy.memory),
                self.enemy_substitute_hp(self.pyboy.memory),
                self.enemy_move_power(self.pyboy.memory),
                self.enemy_move_accuracy(self.pyboy.memory),
                self.player_move_power(self.pyboy.memory),
                self.player_move_accuracy(self.pyboy.memory),
                self.enemy_level(self.pyboy.memory),
                self.pokemon_level(self.pyboy.memory),
                self.enemy_pp_first_slot(self.pyboy.memory),
                self.enemy_pp_second_slot(self.pyboy.memory),
                self.enemy_pp_third_slot(self.pyboy.memory),
                self.enemy_pp_fourth_slot(self.pyboy.memory),
                self.pokemon_pp_first_slot(self.pyboy.memory),
                self.pokemon_pp_second_slot(self.pyboy.memory),
                self.pokemon_pp_third_slot(self.pyboy.memory),
                self.pokemon_pp_fourth_slot(self.pyboy.memory),
            ]
        )
        data_2bytes = self.data_normalizer(
            [
                self.enemy_hp(self.pyboy.memory),
                self.enemy_attack(self.pyboy.memory),
                self.enemy_defense(self.pyboy.memory),
                self.enemy_speed(self.pyboy.memory),
                self.enemy_special(self.pyboy.memory),
                self.pokemon_current_hp(self.pyboy.memory),
                self.pokemon_attack(self.pyboy.memory),
                self.pokemon_defense(self.pyboy.memory),
                self.pokemon_speed(self.pyboy.memory),
                self.pokemon_special(self.pyboy.memory),
            ],
            max=65535,
        )

        data = data_bit + data_byte + data_2bytes

        return data if self.is_battle(self.pyboy.memory) else [0] * len(data)

    def enemy_status(self, memory: PyBoyMemoryView | bytes):
        return self.bits_extractor(memory[0xCFE9], end_bit=6)

    def bits_extractor(self, byte: int, start_bit=0, end_bit=7):
        if start_bit < 0 or end_bit > 7 or start_bit > end_bit:
            raise ValueError("Invalid bit range")

        return [1 if (byte & (1 << i)) else 0 for i in range(start_bit, end_bit + 1)]

    def enemy_base_stats(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xD002 + i] for i in range(5)]

    def battle_status_player(self, memory: PyBoyMemoryView | bytes):
        return (
            self.bits_extractor(memory[0xD062])
            + self.bits_extractor(memory[0xD063])
            + self.bits_extractor(memory[0xD064], 0, 3)
        )

    def is_gym_leader_battle_music_playing(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD05C] & 1

    def critical_hit_flag(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD05E] & 1

    def one_hit_ko_flag(self, memory: PyBoyMemoryView | bytes):
        return 1 if memory[0xD05E] & 2 else 0

    def hooked_pokemon_flag(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD05F] & 1

    def number_of_turns_in_current_battle(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCCD5]

    def players_substitute_hp(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCCD7]

    def enemy_substitute_hp(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCCD8]

    def move_menu_type(self, memory: PyBoyMemoryView | bytes):
        return (
            memory[0xCCDB]
            if self.is_battle(memory) or self.is_dialog(memory) or self.is_menu(memory)
            else 0
        )

    def player_selected_move(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCCDC] if self.is_battle(memory) else 0

    def enemy_selected_move(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCCDD] if self.is_battle(memory) else 0

    def your_move_type(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFD5] if self.is_battle(memory) else 0

    def enemy_move_power(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFCE]

    def enemy_move_type(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFCF] if self.is_battle(memory) else 0

    def enemy_move_accuracy(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFD0]

    def player_move_power(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFD4]

    def player_move_accuracy(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFD6]

    def enemy_hp(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFE6] | memory[0xCFE7] << 8

    def enemy_level(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFE8]

    def enemy_status(self, memory: PyBoyMemoryView | bytes):
        return self.bits_extractor(memory[0xCFE9], end_bit=6)

    def enemy_type1(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFEA] if self.is_battle(memory) else 0

    def enemy_type2(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFEB] if self.is_battle(memory) else 0

    def enemy_move1(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFED] if self.is_battle(memory) else 0

    def enemy_move2(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFEE] if self.is_battle(memory) else 0

    def enemy_move3(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFEF] if self.is_battle(memory) else 0

    def enemy_move4(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFF0] if self.is_battle(memory) else 0

    def enemy_max_hp(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFF4] | (memory[0xCFF5] << 8)

    def enemy_attack(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFF6] | (memory[0xCFF7] << 8)

    def enemy_defense(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFF8] | (memory[0xCFF9] << 8)

    def enemy_speed(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFFA] | (memory[0xCFFB] << 8)

    def enemy_special(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFFC] | (memory[0xCFFD] << 8)

    def enemy_pp_first_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFFE]

    def enemy_pp_second_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xCFFF]

    def enemy_pp_third_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD000]

    def enemy_pp_fourth_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD001]

    def enemy_base_stats(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xD002 + i] for i in range(5)]

    def pokemon_current_hp(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD015] | (memory[0xD016] << 8)

    def pokemon_status(self, memory: PyBoyMemoryView | bytes):
        return self.bits_extractor(memory[0xD018], end_bit=6)

    def pokemon_type1(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD019] if self.is_battle(memory) else 0

    def pokemon_type2(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD01A] if self.is_battle(memory) else 0

    def pokemon_move_first_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD01C] if self.is_battle(memory) else 0

    def pokemon_move_second_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD01D] if self.is_battle(memory) else 0

    def pokemon_move_third_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD01E] if self.is_battle(memory) else 0

    def pokemon_move_fourth_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD01F] if self.is_battle(memory) else 0

    def pokemon_level(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD022]

    def pokemon_max_hp(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD023] | (memory[0xD024] << 8)

    def pokemon_attack(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD025] | (memory[0xD026] << 8)

    def pokemon_defense(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD027] | (memory[0xD028] << 8)

    def pokemon_speed(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD029] | (memory[0xD02A] << 8)

    def pokemon_special(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD02B] | (memory[0xD02C] << 8)

    def pokemon_pp_first_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD02D]

    def pokemon_pp_second_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD02E]

    def pokemon_pp_third_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD02F]

    def pokemon_pp_fourth_slot(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD030]

    def poke_mart_items(self, memory: PyBoyMemoryView | bytes):
        data = [memory[i] for i in range(0xCF7C, 0xCF86)]
        return data if self.is_menu(memory) else [0] * len(data)

    def player_data(self):
        data = self.data_normalizer(
            self.player_pokemons_current_hps(self.pyboy.memory),
            max=0xFFFF,
        )

        data += self.player_pokemons_statuses(self.pyboy.memory)

        data += self.data_normalizer(
            self.player_pokemons_experiences(self.pyboy.memory),
            max=0xFFFFFF,
        )

        data += self.data_normalizer(self.player_pokemons_ivs(self.pyboy.memory))

        data += self.data_normalizer(self.player_pokemons_pps(self.pyboy.memory))

        data += self.data_normalizer(
            self.player_pokemons_levels(self.pyboy.memory),
        )

        data += self.data_normalizer(
            self.player_pokemons_max_hps(self.pyboy.memory),
            max=0xFFFF,
        )

        data += self.data_normalizer(
            self.player_pokemons_attacks(self.pyboy.memory),
            max=0xFFFF,
        )

        data += self.data_normalizer(
            self.player_pokemons_defenses(self.pyboy.memory),
            max=0xFFFF,
        )

        data += self.data_normalizer(
            self.player_pokemons_speeds(self.pyboy.memory),
            max=0xFFFF,
        )

        data += self.data_normalizer(
            self.player_pokemons_specials(self.pyboy.memory),
            max=0xFFFF,
        )

        return data

    def player_pokemons_pps(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[i]
            for x in range(self.__pokemon_count)
            for i in range(
                0xD188 + self.__player_pokemon_size * x,
                0xD18C + self.__player_pokemon_size * x,
            )
        ]

    def player_pokemons_ivs(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[i]
            for x in range(self.__pokemon_count)
            for i in range(
                0xD186 + self.__player_pokemon_size * x,
                0xD188 + self.__player_pokemon_size * x,
            )
        ]

    def player_pokemon_types(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[i]
            for x in range(self.__pokemon_count)
            for i in range(
                0xD170 + self.__player_pokemon_size * x,
                0xD172 + self.__player_pokemon_size * x,
            )
        ]

    def player_pokemons_ids(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD16B + self.__player_pokemon_size * x]
            for x in range(self.__pokemon_count)
        ]

    def player_pokemons_current_hps(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD16C + self.__player_pokemon_size * x]
            | memory[0xD16D + self.__player_pokemon_size * x] << 8
            for x in range(self.__pokemon_count)
        ]

    def player_pokemons_statuses(self, memory: PyBoyMemoryView | bytes = None):
        data = []
        for x in range(self.__pokemon_count):
            data += self.bits_extractor(
                memory[0xD16F + self.__player_pokemon_size * x], end_bit=6
            )

        return data

    def player_pokemons_experiences(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD179 + self.__player_pokemon_size * i]
            | memory[0xD17A + self.__player_pokemon_size * i] << 8
            | memory[0xD17B + self.__player_pokemon_size * i] << 16
            for i in range(self.__pokemon_count)
        ]

    def player_pokemons_levels(self, memory: PyBoyMemoryView | bytes = None):
        return [memory[0xD18C]]

    def player_pokemons_max_hps(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD18D + self.__player_pokemon_size * i]
            | memory[0xD18E + self.__player_pokemon_size * i] << 8
            for i in range(self.__pokemon_count)
        ]

    def player_pokemons_attacks(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD18F + self.__player_pokemon_size * i]
            | memory[0xD190 + self.__player_pokemon_size * i] << 8
            for i in range(self.__pokemon_count)
        ]

    def player_pokemons_defenses(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD191 + self.__player_pokemon_size * i]
            | memory[0xD192 + self.__player_pokemon_size * i] << 8
            for i in range(self.__pokemon_count)
        ]

    def player_pokemons_speeds(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD193 + self.__player_pokemon_size * i]
            | memory[0xD194 + self.__player_pokemon_size * i] << 8
            for i in range(self.__pokemon_count)
        ]

    def player_pokemons_specials(self, memory: PyBoyMemoryView | bytes = None):
        return [
            memory[0xD195 + self.__player_pokemon_size * i]
            | memory[0xD196 + self.__player_pokemon_size * i] << 8
            for i in range(self.__pokemon_count)
        ]

    def pokedex_data(self):
        return self.pokedex_own(self.pyboy.memory) + self.pokedex_seen(
            self.pyboy.memory
        )

    def pokedex_own(self, memory: PyBoyMemoryView | bytes):
        data = bytes(memory[0xD2F7:0xD30A])

        bits: list[int] = []
        for byte in data:
            bits.extend(self.bits_extractor(byte))

        return bits

    def pokedex_seen(self, memory: PyBoyMemoryView | bytes):
        data = memory[0xD30A:0xD31D]

        bits: list[int] = []
        for byte in data:
            bits.extend(self.bits_extractor(byte))

        return bits

    def items_quantities(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xD31F + i * 2] for i in range(20)]

    def items_ids(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xD31E + i * 2] for i in range(20)]

    def player_money(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD347] | (memory[0xD348] << 8) | (memory[0xD349] << 16)

    def badges(self, memory: PyBoyMemoryView | bytes):
        return self.bits_extractor(memory[0xD356])

    def stored_items_ids(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xD53B + 2 * i] for i in range(50)]

    def stored_items_quantities(self, memory: PyBoyMemoryView | bytes):
        return [memory[0xD53C + 2 * i] for i in range(50)]

    def game_coins(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD5A4] | (memory[0xD5A5] << 8)

    def event_flags_data(self, memory: PyBoyMemoryView | bytes):
        return (
            [
                self.starters_back(memory),
                memory[0xD5C0] & 1,
                self.have_town_map(memory),
                self.have_oaks_parcel(memory),
            ]
            + self.fly_anywhere(memory)
            + [
                self.safari_zone_time(memory),
                self.fossilized_pokemon(memory),
                self.position_in_air(memory),
                self.did_you_get_lapras_yet(memory),
                self.debug_new_game(memory),
                self.fought_giovanni_yet(memory),
                self.fought_brock_yet(memory),
                self.fought_misty_yet(memory),
                self.fought_lt_surge_yet(memory),
                self.fought_erika_yet(memory),
                self.fought_articuno_yet(memory),
                self.fought_koga_yet(memory),
                self.fought_blaine_yet(memory),
                self.fought_sabrina_yet(memory),
                self.fought_zapdos_yet(memory),
                self.fought_snorlax_yet_vermilion(memory),
                self.fought_snorlax_yet_celadon(memory),
                self.fought_moltres_yet(memory),
                self.is_ss_anne_here(memory),
                self.mewtwo_can_be_caught(memory),
            ]
        )

    def starters_back(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD5AB] & 1

    def have_town_map(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD5F3] & 1

    def have_oaks_parcel(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD60D] & 1

    def bike_speed(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD700]

    def fly_anywhere(self, memory: PyBoyMemoryView | bytes):
        return self.bits_extractor(memory[0xD70B]) + self.bits_extractor(memory[0xD70C])

    def safari_zone_time(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD70D] | (memory[0xD70E] << 8)

    def fossilized_pokemon(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD710] & 1

    def position_in_air(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD714] & 1

    def did_you_get_lapras_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD72E] & 1

    def debug_new_game(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD732] & 1

    def fought_giovanni_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD751] & 1

    def fought_brock_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD755] & 1

    def fought_misty_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD75E] & 1

    def fought_lt_surge_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD773] & 1

    def fought_erika_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD77C] & 1

    def fought_articuno_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD782] & 1

    def safari_gameover(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD790] & 0x80

    def fought_koga_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD792] & 1

    def fought_blaine_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD79A] & 1

    def fought_sabrina_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD7B3] & 1

    def fought_zapdos_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD7D4] & 1

    def fought_snorlax_yet_vermilion(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD7D8] & 1

    def fought_snorlax_yet_celadon(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD7E0] & 1

    def fought_moltres_yet(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD7EE] & 1

    def is_ss_anne_here(self, memory: PyBoyMemoryView | bytes):
        return memory[0xD803] & 1

    def mewtwo_can_be_caught(self, memory: PyBoyMemoryView | bytes):
        return 1 if memory[0xD85F] & 2 else 0

    def stored_pokemon_data(self, memory: PyBoyMemoryView | bytes):
        data = []
        data = self.data_normalizer(self.stored_pokemon_hps(memory), max=0xFFFF)
        data += self.data_normalizer(self.stored_pokemon_levels(memory))
        data += self.data_normalizer(self.stored_pokemon_statuses(memory))
        data += self.data_normalizer(
            self.stored_pokemon_experiences(memory), max=0xFFFFFF
        )
        data += self.data_normalizer(self.stored_pokemon_pps(memory))

        return data

    def stored_pokemon_pps(self, memory: PyBoyMemoryView | bytes):
        return [
            memory[x + i * self.__stored_pokemon_size]
            for i in range(self.__stored_pokemon_count)
            for x in range(0xDAB3, 0xDAB7)
        ]

    def stored_pokemon_experiences(self, memory: PyBoyMemoryView | bytes):
        return [
            memory[0xDAA4 + i * self.__stored_pokemon_size]
            | (memory[0xDAA5 + i * self.__stored_pokemon_size] << 8)
            | (memory[0xDAA6 + i * self.__stored_pokemon_size] << 16)
            for i in range(self.__stored_pokemon_count)
        ]

    def stored_pokemon_moves(self, memory: PyBoyMemoryView | bytes):
        return [
            memory[x + i * self.__stored_pokemon_size]
            for i in range(self.__stored_pokemon_count)
            for x in range(0xDA9E, 0xDAA2)
        ]

    def stored_pokemon_types(self, memory: PyBoyMemoryView | bytes):
        return [
            memory[x + i * self.__stored_pokemon_size]
            for i in range(self.__stored_pokemon_count)
            for x in range(0xDA9B, 0xDA9D)
        ]

    def stored_pokemon_statuses(self, memory: PyBoyMemoryView | bytes):
        return [
            bit
            for i in range(self.__stored_pokemon_count)
            for bit in self.bits_extractor(
                memory[0xDA9A + i * self.__stored_pokemon_size]
            )
        ]

    def stored_pokemon_levels(self, memory: PyBoyMemoryView | bytes):
        return [
            memory[0xDA99 + i * self.__stored_pokemon_size]
            for i in range(self.__stored_pokemon_count)
        ]

    def stored_pokemon_hps(self, memory: PyBoyMemoryView | bytes):
        return [
            x | (y << 8)
            for x, y in zip(
                [
                    memory[0xDA97 + i * self.__stored_pokemon_size]
                    for i in range(self.__stored_pokemon_count)
                ],
                [
                    memory[0xDA98 + i * self.__stored_pokemon_size]
                    for i in range(self.__stored_pokemon_count)
                ],
            )
        ]

    def stored_pokemon_ids(self, memory: PyBoyMemoryView | bytes):
        return [
            memory[0xDA96 + i * self.__stored_pokemon_size]
            for i in range(self.__stored_pokemon_count)
        ]

    def reward_dialog(self):
        return (
            0.01
            if self.visited_dialogs_count.get(self.dialog_id(self.pyboy.memory), 0)
            < self.visited_dialogs_count_max
            else 0
        )

    def reward_battle(self, memory: bytes):
        reward = 0.0

        reward += self.reward_players_substitute_hp(memory)
        reward += self.reward_enemy_substitute_hp(memory)
        reward += self.reward_enemy_hp(memory)
        reward += self.reward_enemy_status(memory)
        reward += self.reward_pokemon_current_hp(memory)
        reward += self.reward_pokemon_status(memory)
        reward += self.reward_critical_hit_flag(memory)
        reward += self.reward_one_hit_ko_flag(memory)

        return reward

    def reward_position(self):
        return (
            0.01 if self.visited_positions_count.get(self.get_position(), 0) == 0 else 0
        )

    def reward_players_substitute_hp(self, memory: bytes):
        return (
            self.players_substitute_hp(self.pyboy.memory)
            - self.players_substitute_hp(memory)
        ) / 255

    def reward_enemy_substitute_hp(self, memory: bytes):
        return (
            self.enemy_substitute_hp(memory)
            - self.enemy_substitute_hp(self.pyboy.memory)
        ) / 255

    def reward_enemy_hp(self, memory: bytes):
        return (
            (self.enemy_hp(memory) - self.enemy_hp(self.pyboy.memory))
            / self.enemy_max_hp(self.pyboy.memory)
            if self.enemy_max_hp(self.pyboy.memory) != 0
            else 0
        )

    def reward_enemy_status(self, memory: bytes):
        reward = 0

        for bit_before, bit_after in zip(
            self.enemy_status(memory), self.enemy_status(self.pyboy.memory)
        ):
            if bit_before == 0 and bit_after == 1:
                reward += 0.3
            elif bit_before == 1 and bit_after == 0:
                reward -= 0.3

        return reward

    def reward_pokemon_current_hp(self, memory: bytes):
        return (
            (
                self.pokemon_current_hp(self.pyboy.memory)
                - self.pokemon_current_hp(memory)
            )
            / self.pokemon_max_hp(self.pyboy.memory)
            if self.pokemon_max_hp(self.pyboy.memory) != 0
            else 0
        )

    def reward_pokemon_status(self, memory: bytes):
        reward = 0

        for bit_before, bit_after in zip(
            self.pokemon_status(memory), self.pokemon_status(self.pyboy.memory)
        ):
            if bit_before == 0 and bit_after == 1:
                reward -= 0.3
            elif bit_before == 1 and bit_after == 0:
                reward += 0.3

        return reward

    def reward_critical_hit_flag(self, memory: bytes):
        return (
            0.3
            if self.critical_hit_flag(memory) == 0
            and self.critical_hit_flag(self.pyboy.memory) == 1
            else 0.0
        )

    def reward_one_hit_ko_flag(self, memory: bytes):
        return (
            0.3
            if self.one_hit_ko_flag(memory) == 0
            and self.one_hit_ko_flag(self.pyboy.memory) == 1
            else 0.0
        )

    def reward_pokedex(self, memory: bytes):
        return self.reward_pokedex_own(memory) + self.reward_pokedex_seen(memory)

    def reward_pokedex_own(self, memory: bytes):
        reward = 0

        for bit_before, bit_after, visited in zip(
            self.pokedex_own(memory),
            self.pokedex_own(self.pyboy.memory),
            self.visited_pokedex_own,
        ):
            if bit_before == 0 and bit_after == 1 and visited == 0:
                reward += 0.5

        return reward

    def reward_pokedex_seen(self, memory: bytes):
        reward = 0

        for bit_before, bit_after, visited in zip(
            self.pokedex_seen(memory),
            self.pokedex_seen(self.pyboy.memory),
            self.visited_pokedex_seen,
        ):
            if bit_before == 0 and bit_after == 1 and visited == 0:
                reward += 0.4

        return reward

    def reward_badges(self, memory: bytes):
        reward = 0

        for bit_before, bit_after in zip(
            self.badges(memory), self.badges(self.pyboy.memory)
        ):
            if bit_before == 0 and bit_after == 1:
                reward += 1

        return reward

    def reward_event_flags(self, memory: bytes):
        reward = 0

        for flag_x, flag_y in zip(
            self.event_flags_data(memory), self.event_flags_data(self.pyboy.memory)
        ):
            if flag_x == 0 and flag_y == 1:
                reward += 1

        return reward

    def reward_milestones(self, memory: bytes):
        reward = 0.0

        reward += self.reward_badges(memory)

        reward += self.reward_event_flags(memory)

        return reward


: 

In [ ]:
from dataclasses import dataclass
import io
import os
from queue import Queue
from typing import Any

from pyboy import PyBoy
import torch

import keyboard
import time


@dataclass
class Emulator:
    saves = "saves"
    truncated_count_file_name = "truncated_count"
    terminated_count_file_name = "terminated_count"
    buttons = [
        [],
        ["a"],
        ["b"],
        ["start"],
        ["select"],
        ["left"],
        ["right"],
        ["up"],
        ["down"],
    ]
    ticks_per_step = 32
    ALL_BUTTONS = ["a", "b", "start", "select", "left", "right", "up", "down"]

    __use_sdl: bool = False

    @property
    def use_sdl(self) -> bool:
        return self.__use_sdl

    @use_sdl.setter
    def use_sdl(self, use_sdl: bool):
        if use_sdl == self.__use_sdl:
            return

        self.__use_sdl = bool(use_sdl)

        if self.__pyboy is None:
            return

        with io.BytesIO() as f:
            self.pyboy.save_state(f)
            f.seek(0)
            self.pyboy.stop(False)
            self.__pyboy = None
            self.pyboy.load_state(f)

    __pyboy: None | PyBoy = None

    @property
    def pyboy(self):
        if self.__pyboy is None:
            window_str = "SDL2" if self.use_sdl else "null"
            self.__pyboy = PyBoy(f"rom.gb", sound_emulated=False, window=window_str)
            if self.__data is not None:
                self.__data.pyboy = self.__pyboy

        return self.__pyboy

    __data: None | Data = None

    @property
    def data(self):
        if self.__data is None:
            self.__data = Data(pyboy=self.pyboy)

        return self.__data

    def reset(self, dir: str | None = None):
        path = f"{self.saves}/{dir}"

        with open(f"{path}/checkpoint.state", "rb") as f:
            self.pyboy.load_state(f)

        self.data.clean()

        return (bytes(self.pyboy.memory[0:0x10000]), self.data.inputs())

    def step(self, memory: bytes, action: int):
        self.ticks(action)

        reward = self.data.reward(memory)

        terminated = self.data.terminated(memory)

        truncated = self.data.truncated()

        self.data.count(memory, reward)

        if 0 < self.data.reward_event_flags(memory) or 0 < self.data.reward_badges(
            memory
        ):
            self.data.clean()

        return (
            bytes(self.pyboy.memory[0:0x10000]),
            self.data.inputs(),
            reward,
            terminated,
            truncated,
        )

    def auto_mode(self, queue_logs: Queue):
        self.use_sdl = True

        self.pyboy.set_emulation_speed(0)

        memory, inputs = self.reset(dir="start")

        while True:
            action = 0

            key = keyboard.read_key()
            if key == "up":
                action = 7
            elif key == "down":
                action = 8
            elif key == "left":
                action = 5
            elif key == "right":
                action = 6
            elif key == "a":
                action = 1
            elif key == "b":
                action = 2
            elif key == "space":
                action = 3
            elif key == "enter":
                action = 4
            elif key == "q":
                break

            memory, inputs, reward, terminated, truncated = self.step(
                memory=memory, action=action
            )

            if truncated:
                break

            queue_logs.put_nowait("==================================")
            queue_logs.put_nowait(f"Reward: {reward:.2f}")
            queue_logs.put_nowait(f"Terminated: {terminated}")
            queue_logs.put_nowait(f"Truncated: {truncated}")
            queue_logs.put_nowait("==================================")

            time.sleep(0.1)

        self.pyboy.stop(False)

    def ticks(self, action: int):
        for button in self.buttons[action]:
            self.pyboy.button_press(button)

        self.pyboy.tick(self.ticks_per_step / 2)

        for i in range(len(self.ALL_BUTTONS)):
            self.pyboy.button_release(self.ALL_BUTTONS[i])

        self.pyboy.tick(self.ticks_per_step / 2)

    def mask_action(self, action: int) -> int:

        return action

    def evaluate_greedy(
        self,
        model_state_dict: dict[str, Any],
        queue_logs: Queue,
        is_debug: bool,
        is_evaluation_window: bool,
    ):
        self.use_sdl = is_evaluation_window

        model = get_model(device="cpu")
        model.load_state_dict(model_state_dict)
        model.eval()

        total_reward = 0.0

        memory, inputs = self.reset(dir="start")

        while True:
            with torch.inference_mode():
                q = model(inputs)
                q = q.squeeze(0)

            action = int(torch.argmax(q).item())

            next_memory, next_inputs, reward, terminated, truncated = self.step(
                memory=memory, action=action
            )

            if 0 < self.data.reward_event_flags(memory) or 0 < self.data.reward_badges(
                memory
            ):
                self.save_last_checkpoint("saves/last")

            total_reward += reward

            if is_debug:
                queue_logs.put_nowait(
                    f"Action: {action}, Reward: {reward:.2f}, Terminated: {terminated}, Truncated: {truncated}"
                )

            if truncated:
                break

            memory, inputs = (next_memory, next_inputs)

        self.pyboy.stop(False)

        return total_reward

    def save_last_checkpoint(self, path: str):
        os.makedirs(path, exist_ok=True)
        with open(f"{path}/checkpoint.state", "wb") as f:
            self.pyboy.save_state(f)


: 

In [ ]:
import math
import os
import torch
import torch.nn as nn


def get_model(device: str, name: str | None = None):
    emulator = Emulator()

    inputs = emulator.data.inputs()

    continuous_dim = len(inputs["continuous"])

    single_embed_dim = 16 + 16 + 4 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16

    multi_embed_dim = (
        len(inputs["move_id"]) * 16
        + len(inputs["move_type"]) * 16
        + len(inputs["pokemon_id"]) * 16
        + len(inputs["pokemon_type"]) * 16
        + len(inputs["sprite_id"]) * 16
        + len(inputs["item_id"]) * 16
        + len(inputs["sprite_data_movement_statuses"]) * 2
        + len(inputs["sprite_data_facing_directions"]) * 4
        + len(inputs["sprite_data_y_positions"]) * 16
        + len(inputs["sprite_data_x_positions"]) * 16
    )

    total_in_dim = continuous_dim + single_embed_dim + multi_embed_dim

    model = ModelPokemon(
        in_dim=total_in_dim,
        visited_dialogs_count_max=emulator.data.visited_dialogs_count_max + 1,
        visited_maps_count_max=emulator.data.visited_maps_count_max + 1,
        outputs=len(emulator.buttons),
    ).to(device)

    emulator.pyboy.stop(False)

    if name is None:
        return model

    ckpt_path = f"models/{name}.pth"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Model checkpoint not found: {ckpt_path}")

    state = torch.load(ckpt_path, map_location=device)

    model.load_state_dict(
        (
            state["model_state"]
            if isinstance(state, dict) and "model_state" in state
            else state
        ),
        strict=True,
    )

    return model


class ModelPokemon(nn.Module):
    def __init__(
        self,
        in_dim: int,
        visited_dialogs_count_max: int,
        visited_maps_count_max: int,
        outputs: int,
    ):
        super().__init__()

        visited_dialogs_count_output = int(math.sqrt(visited_dialogs_count_max))
        visited_maps_count_output = int(math.sqrt(visited_maps_count_max))
        in_dim = in_dim + visited_dialogs_count_output + visited_maps_count_output

        self.map_id = nn.Embedding(256, 16)
        self.dialog_id = nn.Embedding(256, 16)
        self.index_of_current_pokemon_send_out = nn.Embedding(6, 4)
        self.type_of_battle = nn.Embedding(256, 16)
        self.move_menu_type = nn.Embedding(256, 16)
        self.position_x = nn.Embedding(256, 16)
        self.position_y = nn.Embedding(256, 16)
        self.bike_speed = nn.Embedding(256, 16, padding_idx=0)
        self.menu_position_x = nn.Embedding(256, 16)
        self.menu_position_y = nn.Embedding(256, 16)
        self.current_menu_selected_item = nn.Embedding(256, 16)
        self.visited_dialogs_count = nn.Embedding(
            visited_dialogs_count_max,
            visited_dialogs_count_output,
            padding_idx=0,
        )
        self.visited_maps_count = nn.Embedding(
            visited_maps_count_max,
            visited_maps_count_output,
            padding_idx=0,
        )

        self.move_id = nn.Embedding(256, 16, padding_idx=0)
        self.move_type = nn.Embedding(256, 16, padding_idx=0)
        self.pokemon_id = nn.Embedding(256, 16, padding_idx=0)
        self.pokemon_type = nn.Embedding(256, 16, padding_idx=0)
        self.sprite_id = nn.Embedding(256, 16, padding_idx=0)
        self.item_id = nn.Embedding(256, 16, padding_idx=0)
        self.sprite_data_movement_statuses = nn.Embedding(4, 2)
        self.sprite_data_facing_directions = nn.Embedding(13, 4)
        self.sprite_data_y_positions = nn.Embedding(256, 16, padding_idx=0)
        self.sprite_data_x_positions = nn.Embedding(256, 16, padding_idx=0)

        self.fc = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 4096),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(4096, 2048),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(2048, 1024),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.Linear(128, outputs),
        )

    def _as_float_batch(self, t, device):
        t = t.to(device)
        if t.dim() == 1:
            t = t.unsqueeze(0)
        return t.float()

    def _as_long_scalar_batch(self, t, device):
        t = t.to(device)
        if t.dtype != torch.long:
            t = t.long()
        if t.dim() == 0:
            t = t.unsqueeze(0)
        return t

    def _as_long_seq_batch(self, t, device):
        t = t.to(device)
        if t.dtype != torch.long:
            t = t.long()
        if t.dim() == 1:
            t = t.unsqueeze(0)
        return t

    def forward(self, x):
        device = next(self.parameters()).device

        cont = self._as_float_batch(x["continuous"], device)

        map_id_emb = self.map_id(self._as_long_scalar_batch(x["map_id"], device))
        dialog_id_emb = self.dialog_id(
            self._as_long_scalar_batch(x["dialog_id"], device)
        )
        index_emb = self.index_of_current_pokemon_send_out(
            self._as_long_scalar_batch(x["index_of_current_pokemon_send_out"], device)
        )
        type_battle_emb = self.type_of_battle(
            self._as_long_scalar_batch(x["type_of_battle"], device)
        )
        move_menu_emb = self.move_menu_type(
            self._as_long_scalar_batch(x["move_menu_type"], device)
        )
        position_x_emb = self.position_x(
            self._as_long_scalar_batch(x["position_x"], device)
        )
        position_y_emb = self.position_y(
            self._as_long_scalar_batch(x["position_y"], device)
        )
        bike_speed_emb = self.bike_speed(
            self._as_long_scalar_batch(x["bike_speed"], device)
        )
        menu_position_x_emb = self.menu_position_x(
            self._as_long_scalar_batch(x["menu_position_x"], device)
        )
        menu_position_y_emb = self.menu_position_y(
            self._as_long_scalar_batch(x["menu_position_y"], device)
        )
        current_menu_selected_item_emb = self.current_menu_selected_item(
            self._as_long_scalar_batch(x["current_menu_selected_item"], device)
        )
        visited_dialogs_count_emb = self.visited_dialogs_count(
            self._as_long_scalar_batch(x["visited_dialogs_count"], device)
        )
        visited_maps_count_emb = self.visited_maps_count(
            self._as_long_scalar_batch(x["visited_maps_count"], device)
        )

        move_id_full = self.move_id(self._as_long_seq_batch(x["move_id"], device))
        move_id_emb = move_id_full.reshape(move_id_full.size(0), -1)

        move_type_full = self.move_type(self._as_long_seq_batch(x["move_type"], device))
        move_type_emb = move_type_full.reshape(move_type_full.size(0), -1)

        pokemon_id_full = self.pokemon_id(
            self._as_long_seq_batch(x["pokemon_id"], device)
        )
        pokemon_id_emb = pokemon_id_full.reshape(pokemon_id_full.size(0), -1)

        pokemon_type_full = self.pokemon_type(
            self._as_long_seq_batch(x["pokemon_type"], device)
        )
        pokemon_type_emb = pokemon_type_full.reshape(pokemon_type_full.size(0), -1)

        sprite_id_full = self.sprite_id(self._as_long_seq_batch(x["sprite_id"], device))
        sprite_id_emb = sprite_id_full.reshape(sprite_id_full.size(0), -1)

        item_id_full = self.item_id(self._as_long_seq_batch(x["item_id"], device))
        item_id_emb = item_id_full.reshape(item_id_full.size(0), -1)

        sprite_data_movement_statuses_full = self.sprite_data_movement_statuses(
            self._as_long_seq_batch(x["sprite_data_movement_statuses"], device)
        )
        sprite_data_movement_statuses_emb = sprite_data_movement_statuses_full.reshape(
            sprite_data_movement_statuses_full.size(0), -1
        )

        sprite_data_facing_directions_full = self.sprite_data_facing_directions(
            self._as_long_seq_batch(x["sprite_data_facing_directions"], device)
        )
        sprite_data_facing_directions_emb = sprite_data_facing_directions_full.reshape(
            sprite_data_facing_directions_full.size(0), -1
        )

        sprite_data_y_positions_full = self.sprite_data_y_positions(
            self._as_long_seq_batch(x["sprite_data_y_positions"], device)
        )
        sprite_data_y_positions_emb = sprite_data_y_positions_full.reshape(
            sprite_data_y_positions_full.size(0), -1
        )

        sprite_data_x_positions_full = self.sprite_data_x_positions(
            self._as_long_seq_batch(x["sprite_data_x_positions"], device)
        )
        sprite_data_x_positions_emb = sprite_data_x_positions_full.reshape(
            sprite_data_x_positions_full.size(0), -1
        )

        h = torch.cat(
            [
                cont,
                map_id_emb,
                dialog_id_emb,
                index_emb,
                type_battle_emb,
                move_menu_emb,
                position_x_emb,
                position_y_emb,
                bike_speed_emb,
                menu_position_x_emb,
                menu_position_y_emb,
                current_menu_selected_item_emb,
                visited_dialogs_count_emb,
                visited_maps_count_emb,
                move_id_emb,
                move_type_emb,
                pokemon_id_emb,
                pokemon_type_emb,
                sprite_id_emb,
                item_id_emb,
                sprite_data_movement_statuses_emb,
                sprite_data_facing_directions_emb,
                sprite_data_y_positions_emb,
                sprite_data_x_positions_emb,
            ],
            dim=1,
        )

        return self.fc(h)


: 

In [ ]:
from collections import deque
from dataclasses import dataclass
from multiprocessing import Queue
from multiprocessing.sharedctypes import Synchronized
import os
from queue import Full
import random
import time
import traceback
from typing import Any
import torch
import sys


@dataclass
class ExperienceWorker:
    queue_logs: Queue
    queue_data: Queue
    window: Synchronized
    gamma: float
    model_state_dict: dict[str, Any]
    epsilon: float
    td_error_steps = 5
    start_save_chance = 0.5

    __last_save_path = "last"

    @property
    def last_save_path(self):
        return (
            self.__last_save_path
            if os.path.exists(f"saves/{self.__last_save_path}")
            else "start"
        )

    __model: None | ModelPokemon = None

    @property
    def model(self):
        if not self.__model:
            self.__model = get_model(device="cpu")

            self.__model.load_state_dict(self.model_state_dict)

            self.__model.eval()

        return self.__model

    __buffer: deque | None = None

    @property
    def buffer(self):
        if self.__buffer is None:
            self.__buffer = deque(maxlen=self.td_error_steps)

        return self.__buffer

    __emulator: Emulator | None = None

    @property
    def emulator(self):
        if self.__emulator is None:
            self.__emulator = Emulator()

        return self.__emulator

    def run(self):
        try:
            os.chdir("/content/drive/MyDrive/pokemon")
            traceback.print_exc()
            sys.stdout.flush()
            sys.stderr.flush()
            memory, inputs = self.emulator.reset(
                dir=(
                    "start"
                    if random.random() < self.start_save_chance
                    else self.last_save_path
                )
            )

            while True:
                action = self.get_action(inputs)

                next_memory, next_inputs, reward, terminated, truncated = (
                    self.emulator.step(memory=memory, action=action)
                )

                self.buffer.append(
                    {
                        "inputs": self.detach_to_cpu(inputs),
                        "action": action,
                        "next_inputs": self.detach_to_cpu(next_inputs),
                        "reward": reward,
                    }
                )

                self.put_to_queue_data(terminated=terminated, truncated=truncated)

                if truncated:
                    break

                memory, inputs = next_memory, next_inputs

                self.emulator.use_sdl = bool(self.window.get())

        except Exception as e:
            self.queue_logs.put_nowait(f"{e}\n{traceback.print_exc()}")
        finally:
            self.emulator.pyboy.stop(False)
            self.queue_logs.put_nowait("Worker stopped.")

    def get_action(self, inputs: dict[float]):
        if random.random() < self.epsilon:
            action = random.randint(0, len(self.emulator.buttons) - 1)
        else:
            with torch.inference_mode():
                q = self.model(inputs)
                q = q.squeeze(0)

            action = int(torch.argmax(q).item())

        return self.emulator.mask_action(action)

    def put_to_queue_data(self, terminated: bool, truncated: bool):
        if self.buffer.maxlen <= len(self.buffer):
            while len(self.buffer):
                reward, discount = 0.0, 1.0

                for item in self.buffer:
                    reward += discount * item["reward"]
                    discount *= self.gamma

                try:
                    self.queue_data.put_nowait(
                        (
                            self.detach_to_cpu(self.buffer[0]["inputs"]),
                            self.buffer[0]["action"],
                            self.detach_to_cpu(self.buffer[-1]["next_inputs"]),
                            reward,
                            terminated,
                            truncated,
                            len(self.buffer),
                        )
                    )
                except Full:
                    time.sleep(0.01)
                    pass

                if truncated:
                    self.buffer.popleft()
                else:
                    break

    def detach_to_cpu(self, inputs):
        out = {}

        for k, v in inputs.items():
            if torch.is_tensor(v):
                v = v.detach().cpu()

                if k == "continuous":
                    out[k] = v.numpy().copy()
                else:
                    if v.numel() == 1:
                        out[k] = int(v.item())
                    else:
                        out[k] = v.tolist()
            else:
                out[k] = v

        return out


: 

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from multiprocessing import Queue, Manager
import multiprocessing as mp
from multiprocessing.sharedctypes import Synchronized
from multiprocessing.synchronize import Event
import os
from threading import RLock, Thread
from time import sleep
import traceback
from typing import Any
import torch.optim as optim
import numpy as np
import torch
from math import inf


@dataclass
class TrainWorker:
    max_workers: int = field(default_factory=lambda: 6)
    device: str = field(
        default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu"
    )
    queue_data: Queue = field(default_factory=lambda: Manager().Queue())
    event_start: Event = field(default_factory=lambda: Manager().Event())
    queue_logs: Queue = field(default_factory=lambda: Manager().Queue())
    model_lock: RLock = field(default_factory=lambda: Manager().RLock())
    is_debug: Synchronized = field(default_factory=lambda: Manager().Value("b", False))
    buffer: PrioritizedReplayBuffer = field(
        default_factory=lambda: PrioritizedReplayBuffer(capacity=500000)
    )
    buffer_lock: RLock = field(default_factory=lambda: Manager().RLock())
    is_evaluation_window: Synchronized = field(
        default_factory=lambda: Manager().Value("b", False)
    )
    train_use_sdl: Synchronized = field(
        default_factory=lambda: Manager().Value("b", False)
    )

    batch_size = 512
    grad_accum_steps = 1
    lr = 0.0001
    weight_decay = 1e-5
    gamma = 0.99
    criterion: torch.nn.SmoothL1Loss = field(default_factory=torch.nn.SmoothL1Loss)
    tau = 0.005
    epsilon = 0.3
    # loss tracking
    running_loss_ema: float = 0.0
    loss_ema_alpha: float = 0.001
    last_loss: float = 0.0
    target_update_interval = 1000
    _opt_steps: int = 0

    per_alpha: float = 0.6
    per_beta_start: float = 0.4
    per_beta_frames: int = 100000

    count: int = 0

    __model: None | ModelPokemon = None

    @property
    def model(self):
        if not self.__model:
            name = None
            if os.path.exists("models/latest.pth"):
                name = "latest"
            elif os.path.exists("models/best.pth"):
                name = "best"

            self.__model = get_model(device=self.device, name=name)

            self.__model.train()

        return self.__model

    __target_model: None | ModelPokemon = None

    @property
    def target_model(self):
        if not self.__target_model:
            self.__target_model = get_model(self.device)

            with self.model_lock:
                self.__target_model.load_state_dict(self.model.state_dict())

            self.__target_model.eval()

        return self.__target_model

    __optimizer: None | optim.AdamW = None

    @property
    def optimizer(self):
        if not self.__optimizer:
            with self.model_lock:
                self.__optimizer = optim.AdamW(
                    self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay
                )

        return self.__optimizer

    __gamma_tensor: torch.Tensor | None = None

    @property
    def gamma_tensor(self):
        if not self.__gamma_tensor:
            self.__gamma_tensor = torch.tensor(
                self.gamma, device=self.device, dtype=torch.float32
            )

        return self.__gamma_tensor

    __best_eval_return: float | None = None

    @property
    def best_eval_return(self):
        if self.__best_eval_return is None:
            try:
                model_best = torch.load(
                    "models/best.pth", map_location=torch.device("cpu")
                )

                self.__best_eval_return = float(
                    model_best["best_return"]
                    if isinstance(model_best, dict) and "best_return" in model_best
                    else -float("inf")
                )

            except Exception:
                self.__best_eval_return = -inf

        return self.__best_eval_return

    @best_eval_return.setter
    def best_eval_return(self, best_eval_return: float):
        self.__best_eval_return = best_eval_return

    __run_queue_thread: Thread | None = None

    @property
    def run_queue_thread(self):
        if not self.__run_queue_thread:
            self.__run_queue_thread = Thread(target=self.run_queue, daemon=True)

        return self.__run_queue_thread

    @run_queue_thread.setter
    def run_queue_thread(self, value: Thread | None):
        self.__run_queue_thread = value

    __run_train_thread: Thread | None = None

    @property
    def run_train_thread(self):
        if not self.__run_train_thread:
            self.__run_train_thread = Thread(target=self.run_train, daemon=True)

        return self.__run_train_thread

    @run_train_thread.setter
    def run_train_thread(self, value: Thread | None):
        self.__run_train_thread = value

    __run_workers_thread: Thread | None = None

    @property
    def run_workers_thread(self):
        if not self.__run_workers_thread:
            self.__run_workers_thread = Thread(target=self.run_workers, daemon=True)

        return self.__run_workers_thread

    @run_workers_thread.setter
    def run_workers_thread(self, value: Thread | None):
        self.__run_workers_thread = value

    __run_evaluate_thread: Thread | None = None

    @property
    def run_evaluate_thread(self):
        if not self.__run_evaluate_thread:
            self.__run_evaluate_thread = Thread(target=self.run_evaluate, daemon=True)

        return self.__run_evaluate_thread

    @run_evaluate_thread.setter
    def run_evaluate_thread(self, value: Thread | None):
        self.__run_evaluate_thread = value

    def run(self):
        try:
            self.queue_logs.put_nowait("TrainWorker starting up.")

            self.event_start.set()

            if not self.run_queue_thread.is_alive():
                self.run_queue_thread = None
                self.run_queue_thread.start()

            if not self.run_train_thread.is_alive():
                self.run_train_thread = None
                self.run_train_thread.start()

            if not self.run_workers_thread.is_alive():
                self.run_workers_thread = None
                self.run_workers_thread.start()

            if not self.run_evaluate_thread.is_alive():
                self.run_evaluate_thread = None
                self.run_evaluate_thread.start()
        except Exception as e:
            self.queue_logs.put_nowait(f"{e}\n{traceback.print_exc()}")
            self.event_start.clear()
        finally:
            self.queue_logs.put_nowait("TrainWorker setted up.")

    def run_train(self):
        try:
            self.queue_logs.put_nowait("Training started.")

            while self.event_start.is_set():
                self.optimize_batch()

                if self.is_debug.value and self.count % 10 == 0:
                    with self.buffer_lock:
                        self.queue_logs.put_nowait(
                            f"Count: {self.count} | Buffer: {len(self.buffer)} | Epsilon: {self.epsilon:.2f} | Loss: {self.last_loss:.6f} | EMA Loss: {self.running_loss_ema:.6f}"
                        )

                self.count += 1
        except Exception as e:
            self.queue_logs.put_nowait(f"{e}\n{traceback.print_exc()}")
        finally:
            self.event_start.clear()
            self.queue_logs.put_nowait("Train stopped.")

    def run_workers(self):
        try:
            self.queue_logs.put_nowait("Workers started.")

            while self.event_start.is_set():
                with self.model_lock:
                    model_state_dict = self.model.state_dict()

                with ProcessPoolExecutor(
                    max_workers=self.max_workers, mp_context=mp.get_context("spawn")
                ) as pool:
                    futures = [
                        pool.submit(
                            ExperienceWorker(
                                queue_logs=self.queue_logs,
                                queue_data=self.queue_data,
                                gamma=self.gamma,
                                model_state_dict=model_state_dict,
                                epsilon=self.epsilon,
                                window=self.train_use_sdl,
                            ).run
                        )
                        for _ in range(self.max_workers)
                    ]

                    for future in futures:
                        future.result()

        except Exception as e:
            self.queue_logs.put_nowait(f"{e}\n{traceback.print_exc()}")
        finally:
            self.event_start.clear()
            self.queue_logs.put_nowait("Workers stopped.")

    def run_queue(self):
        try:
            self.queue_logs.put_nowait("Queue handler started.")

            while self.event_start.is_set():
                with self.buffer_lock:
                    self.buffer.add(self.queue_data.get())
        except Exception as e:
            self.queue_logs.put_nowait(f"{e}\n{traceback.print_exc()}")
        finally:
            self.event_start.clear()
            self.queue_logs.put_nowait("Queue handler stopped.")

    def run_evaluate(self):
        try:
            self.queue_logs.put_nowait(f"Starting evaluation.")

            while self.event_start.is_set():
                with self.model_lock:
                    model_state_dict = self.model.state_dict()

                self.save_latest()

                avg_ret = Emulator().evaluate_greedy(
                    model_state_dict=model_state_dict,
                    queue_logs=self.queue_logs,
                    is_debug=False,
                    is_evaluation_window=self.is_evaluation_window.value,
                )

                if self.best_eval_return < avg_ret:
                    self.save_best(avg_ret)

                self.queue_logs.put_nowait(f"Finished evaluation {avg_ret:.2f}.")

        except Exception as e:
            self.queue_logs.put_nowait(f"{e}\n{traceback.print_exc()}")
        finally:
            self.event_start.clear()
            self.queue_logs.put_nowait("Evaluation stopped.")

    def optimize_batch(self):
        with self.buffer_lock:
            if len(self.buffer) < self.batch_size:
                sleep(0.1)
                return

        with self.buffer_lock:
            batch, idxs, weights = self.buffer.sample(
                self.batch_size,
                min(
                    1.0,
                    self.per_beta_start
                    + self.count * (1.0 - self.per_beta_start) / self.per_beta_frames,
                ),
            )
        weights = torch.tensor(weights, device=self.device, dtype=torch.float32)

        (inputs, actions, next_inputs, rewards, terminateds, truncateds, steps) = zip(
            *batch
        )

        inputs = self.collate_states(inputs)
        actions = torch.tensor(actions, device=self.device, dtype=torch.long)
        next_inputs = self.collate_states(next_inputs)
        rewards = torch.tensor(rewards, device=self.device, dtype=torch.float32)
        terminateds = torch.tensor(terminateds, device=self.device, dtype=torch.bool)
        truncateds = torch.tensor(truncateds, device=self.device, dtype=torch.bool)
        steps = torch.tensor(steps, device=self.device, dtype=torch.long)

        micro = self.batch_size // self.grad_accum_steps
        assert self.batch_size % self.grad_accum_steps == 0

        self.optimizer.zero_grad(set_to_none=True)
        batch_loss = 0.0

        total_td_errors = np.zeros(self.batch_size)

        for i in range(self.grad_accum_steps):
            sl = slice(i * micro, (i + 1) * micro)
            s = {k: v[sl] for k, v in inputs.items()}
            ns = {k: v[sl] for k, v in next_inputs.items()}
            a = actions[sl]
            rN = rewards[sl]
            te = terminateds[sl]
            tr = truncateds[sl]
            n = steps[sl]
            w_slice = weights[sl]

            with self.model_lock:
                q_all = self.model(s)
            q_sa = q_all.gather(1, a.view(-1, 1)).squeeze(1)

            with torch.no_grad():
                with self.model_lock:
                    was_training = self.model.training
                    self.model.eval()
                    next_q_online = self.model(ns)
                    next_a = torch.argmax(next_q_online, dim=1)
                    if was_training:
                        self.model.train()

                    next_q_target = (
                        self.target_model(ns).gather(1, next_a.view(-1, 1)).squeeze(1)
                    )

                gamma_pow_n = torch.pow(self.gamma_tensor, n)
                bootstrap_mask = (~te).float()
                target = rN + bootstrap_mask * gamma_pow_n * next_q_target

            td_error = torch.abs(q_sa - target).detach()
            total_td_errors[sl] = td_error.cpu().numpy()

            element_wise_loss = torch.nn.functional.smooth_l1_loss(
                q_sa, target, reduction="none"
            )
            weighted_loss = (element_wise_loss * w_slice).mean()

            loss = weighted_loss / self.grad_accum_steps
            loss.backward()
            batch_loss += float(loss.detach().item())

        with self.model_lock:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

        self.optimizer.step()

        self.buffer.update_priorities(idxs, total_td_errors)

        self._opt_steps += 1
        if self._opt_steps % self.target_update_interval == 0:
            self.hard_update_target()

        self.soft_update_target()

        self.last_loss = batch_loss
        if self.running_loss_ema == 0.0:
            self.running_loss_ema = batch_loss
        else:
            self.running_loss_ema = (
                self.loss_ema_alpha * batch_loss
                + (1 - self.loss_ema_alpha) * self.running_loss_ema
            )

    def collate_states(self, list_of_dicts: tuple[Any, ...]):
        batch = {}
        keys = list(list_of_dicts[0].keys())
        for k in keys:
            vals = [d[k] for d in list_of_dicts]
            if k == "continuous":
                t = torch.from_numpy(np.stack(vals, axis=0)).float()
                batch[k] = t.to(self.device, non_blocking=True)
            else:
                batch[k] = torch.tensor(vals, dtype=torch.long, device=self.device)

        return batch

    def soft_update_target(self):
        with torch.no_grad(), self.model_lock:
            for p, tp in zip(self.model.parameters(), self.target_model.parameters()):
                tp.data.copy_(self.tau * p.data + (1 - self.tau) * tp.data)

    def hard_update_target(self):
        with torch.no_grad(), self.model_lock:
            for p, tp in zip(self.model.parameters(), self.target_model.parameters()):
                tp.data.copy_(p.data)

    def save_latest(self):
        os.makedirs("models", exist_ok=True)
        with self.model_lock:
            torch.save(
                {
                    "model_state": self.model.state_dict(),
                    "optimizer_state": self.optimizer.state_dict(),
                    "target_state": self.target_model.state_dict(),
                },
                "models/latest.pth",
            )

    def save_best(self, avg_return: float):
        self.best_eval_return = avg_return

        os.makedirs("models", exist_ok=True)
        with self.model_lock:
            torch.save(
                {
                    "best_return": avg_return,
                    "model_state": self.model.state_dict(),
                },
                "models/best.pth",
            )


: 

In [ ]:
import json
from multiprocessing import Manager, Process
import os
from typing import Any


class TrainModel:
    process: None | Process = None
    __FILE_SETTINGS = "settings.json"
    evaluate_process: None | Process = None
    auto_mode_process: None | Process = None

    def __init__(self):
        self.train_worker = TrainWorker(
            queue_logs=Manager().Queue(),
            queue_data=Manager().Queue(),
            event_start=Manager().Event(),
            is_debug=Manager().Value("b", False),
            is_evaluation_window=Manager().Value("b", False),
            train_use_sdl=Manager().Value("b", False),
        )

    @property
    def settings(self) -> dict[str, Any]:
        if not os.path.isfile(self.__FILE_SETTINGS):
            with open(
                self.__FILE_SETTINGS,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump({}, file, indent=4, ensure_ascii=False)

        with open(
            self.__FILE_SETTINGS,
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    @settings.setter
    def settings(self, settings: dict[str, Any]):
        with open(
            os.path.join(self.__FILE_SETTINGS),
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(settings, file, indent=4, ensure_ascii=False)

    def start(self):
        if self.process is not None and self.process.is_alive():
            raise RuntimeError(f"Worker (pid={self.process.pid}) is already running")

        self.train_worker.run()

    def start_evaluation(self, best_model: bool):
        if self.evaluate_process is not None and self.evaluate_process.is_alive():
            raise RuntimeError(
                f"Evaluation Worker (pid={self.evaluate_process.pid}) is already running"
            )

        self.evaluate_process = Process(
            target=Emulator().evaluate_greedy,
            kwargs={
                "model_state_dict": get_model(
                    "cpu", "best" if best_model else "latest"
                ).state_dict(),
                "queue_logs": self.train_worker.queue_logs,
                "is_debug": self.train_worker.is_debug.value,
                "is_evaluation_window": self.train_worker.is_evaluation_window,
            },
        )

        self.evaluate_process.start()

    def start_auto_mode(self):
        if self.auto_mode_process is not None and self.auto_mode_process.is_alive():
            raise RuntimeError(
                f"Auto Mode Worker (pid={self.auto_mode_process.pid}) is already running"
            )

        self.auto_mode_process = Process(
            target=Emulator().auto_mode,
            kwargs={"queue_logs": self.train_worker.queue_logs},
        )

        self.auto_mode_process.start()


: 

In [ ]:
import queue
from time import sleep
import torch
import os
from multiprocessing import set_start_method, freeze_support
import torch.multiprocessing as tmp

import torch

def main():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision("high")

    set_start_method("spawn", force=True)
    tmp.set_start_method("spawn", force=True)
    freeze_support()

    os.environ.setdefault("CUDA_DEVICE_MAX_CONNECTIONS", "32")

    trainWorker = TrainWorker()

    trainWorker.run()
    try:
        while trainWorker.event_start.is_set():
            try:
                print(trainWorker.queue_logs.get_nowait())
            except queue.Empty:
                sleep(0.1)
    except KeyboardInterrupt:
        pass
    finally:
        trainWorker.event_start.clear()


if __name__ == "__main__":
    main()


: 